<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Information Retrieval with Lakebase Search
#### Duration: 30 minutes
## Overview
In this lab you will work with **Lakebase Search** on a **Lakebase Postgres** branch that has been pre-loaded with Airbnb listing document chunks (with pre-computed **1024-dimensional embeddings** and **tsvector** values). You will use copyable SQL to explore three search workloads: **vector similarity**, **keyword (tsvector)**, and a **combined hybrid** query.

This gives you hands-on practice with the Lakebase retrieval workflow — connecting to a pre-provisioned database and running search queries.

## Learning Objectives
By the end of this lab, you will be able to:
1. **Connect** to a pre-provisioned Lakebase project using the `lb_connect()` factory
2. **Run** vector similarity search using cosine distance (`<=>`) over pgvector embeddings
3. **Run** keyword search using pre-computed `tsvector` columns and `ts_rank`
4. **Run** hybrid search combining vector and keyword signals with weighted scoring
5. **Compare** the results of different search strategies

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
# Get the current user's username
username = spark.sql("SELECT current_user()").first()[0]
print(f"{username.split('@')[0].replace(".","-").replace("_","-")}-lb-search")

### A1. Lakebase Project Setup
Before running the classroom setup, we will need to do 2 things: 
1. Create a Lakebase Project
    - Navigate to the waffle at the top right, click on it, right click on **Lakebase Postgres** and select **Open Link in New Tab**.
    - On the next screen, click **⊕ New Project**
    - Give it the name `<labusername>-lb-search`. Running the previous cell will output what you should copy and paste into the name field.  
        - Example: `labuser_123abc-lb-search`
    - Choose Postgres version 18 (other version may work but this course has been tested on 18)
    - Click **Create**

2. Turn on Lakebase Search
    - Navigate into your Lakebase Project and click on **Settings**
    - Scroll down to the very bottom and click **Enable Lakebase Search** and confirm to enable Lakebase Search

3. Navigate back to the workspace, but leave an additional tab open so we can demonstrate different search methods.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <strong style="color: #c62828; font-size: 1.1em;">Important</strong>
  <div style="color: #333;">

You must name your Lakebase project correctly, or the classroom setup below will not run.

If you see the following error, your Lakebase project was named incorrectly.


```
RuntimeError: 
====================================================================
Lakebase project not found: labuser-xxx-lb-search
```

  </div>
</div>


### A2. Classroom Setup
Run the cell below to set up your catalog, schema, and other classroom assets needed to this notebook.

In [0]:
%run ./Includes/Classroom-Setup-03

### A3. Lakebase Tables: Under the Hood
The previous setup tasks did two important things for you:
1. Create the Lakebase Project with the UI and enabled the **Lakebase Search** feature

2. Populated our project with a table called **doc_chunks** in the **databricks_postgres.public** schema.

This was performed for you to bypass some heavy code, but it's worth understanding different methods for getting these tables situated. We summarize them below, but the technical details go beyond the scope of this course. Please see our [additional trainings](https://www.databricks.com/training/catalog?search=lakebase) for a deeper dive into these concepts as well as our docs pages ([AWS](https://docs.databricks.com/aws/en/oltp/projects/lakebase-search) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/lakebase-search))

<br>

<div class="lbss">
  <style>
    .lbss{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:1120px;margin:0 auto;color:#1B3139;}
    .lbss *{box-sizing:border-box;}
    .lbss code{font-family:ui-monospace,Menlo,Consolas,monospace;}
    .lbss .mono{font-family:ui-monospace,Menlo,Consolas,monospace;background:#EEEDE9;border-radius:5px;padding:2px 8px;font-size:13.5px;color:#0b2026;white-space:nowrap;}
    .lbss .head{text-align:center;margin-bottom:6px;}
    .lbss .h1{font-size:26px;font-weight:800;color:#FF3621;}
    .lbss .sub{font-size:15px;color:#5A6A70;margin-top:4px;}
    .lbss .eyebrow{font-size:12px;font-weight:800;letter-spacing:.6px;text-transform:uppercase;color:#5A6A70;margin:22px 0 10px;}
    .lbss .ing{display:flex;gap:18px;flex-wrap:nowrap;}
    .lbss .card{flex:1 1 0;min-width:0;background:#fff;border:1.5px solid #C6DCEA;border-radius:14px;padding:16px 18px 18px;box-shadow:0 2px 8px rgba(27,49,57,.07);}
    .lbss .bar{height:5px;border-radius:5px;margin:-4px 0 12px;}
    .lbss .cap{font-size:16px;font-weight:800;}
    .lbss .flow{display:flex;align-items:center;gap:10px;margin:12px 0 10px;flex-wrap:wrap;}
    .lbss .arrow{color:#9DB0BA;font-size:18px;font-weight:800;}
    .lbss .note{font-size:12.5px;color:#5A6A70;line-height:1.45;}
    .lbss .tag{font-size:11px;color:#5A6A70;}
    .lbss>input{position:absolute;opacity:0;pointer-events:none;}
    .lbss .tabs{display:flex;gap:12px;flex-wrap:nowrap;}
    .lbss .btn{flex:1 1 0;min-width:0;display:block;cursor:pointer;background:#fff;border:1.5px solid #C6DCEA;border-radius:12px;padding:13px 15px;transition:all .12s;}
    .lbss .btn .n{display:inline-flex;width:22px;height:22px;border-radius:50%;background:#EEF1F3;color:#5A6A70;font-size:12px;font-weight:800;align-items:center;justify-content:center;margin-bottom:7px;}
    .lbss .btn .t{font-size:15px;font-weight:800;color:#1B3139;line-height:1.2;}
    .lbss .btn .w{font-size:12px;color:#5A6A70;margin-top:3px;line-height:1.35;}
    .lbss .btn:hover{border-color:#FF5F46;}
    .lbss .panel{display:none;background:#fff;border:1.5px solid #FF5F46;border-radius:14px;padding:18px 20px;margin-top:14px;box-shadow:0 4px 14px rgba(255,95,70,.12);}
    .lbss .step{display:flex;gap:12px;margin-bottom:12px;}
    .lbss .step:last-child{margin-bottom:0;}
    .lbss .sn{flex:none;width:24px;height:24px;border-radius:50%;background:#FF5F46;color:#fff;font-size:13px;font-weight:800;display:flex;align-items:center;justify-content:center;}
    .lbss .sd{font-size:14px;color:#1B3139;line-height:1.45;padding-top:2px;}
    .lbss #lbss-m1:checked ~ .tabs label[for=lbss-m1],
    .lbss #lbss-m2:checked ~ .tabs label[for=lbss-m2],
    .lbss #lbss-m3:checked ~ .tabs label[for=lbss-m3]{background:#FF5F46;border-color:#FF5F46;box-shadow:0 4px 12px rgba(255,95,70,.22);}
    .lbss #lbss-m1:checked ~ .tabs label[for=lbss-m1] .t,
    .lbss #lbss-m2:checked ~ .tabs label[for=lbss-m2] .t,
    .lbss #lbss-m3:checked ~ .tabs label[for=lbss-m3] .t,
    .lbss #lbss-m1:checked ~ .tabs label[for=lbss-m1] .w,
    .lbss #lbss-m2:checked ~ .tabs label[for=lbss-m2] .w,
    .lbss #lbss-m3:checked ~ .tabs label[for=lbss-m3] .w{color:#fff;}
    .lbss #lbss-m1:checked ~ .tabs label[for=lbss-m1] .n,
    .lbss #lbss-m2:checked ~ .tabs label[for=lbss-m2] .n,
    .lbss #lbss-m3:checked ~ .tabs label[for=lbss-m3] .n{background:rgba(255,255,255,.28);color:#fff;}
    .lbss #lbss-m1:checked ~ .panels #lbss-p1,
    .lbss #lbss-m2:checked ~ .panels #lbss-p2,
    .lbss #lbss-m3:checked ~ .panels #lbss-p3{display:block;}
  </style>

  <div class="eyebrow">Click each card to view different methods</div>
  <input type="radio" name="lbss-m" id="lbss-m1" checked>
  <input type="radio" name="lbss-m" id="lbss-m2">
  <input type="radio" name="lbss-m" id="lbss-m3">
  <div class="tabs">
    <label class="btn" for="lbss-m1"><span class="n">1</span><div class="t">Start from scratch</div><div class="w">No table yet &mdash; build it in Lakebase</div></label>
    <label class="btn" for="lbss-m2"><span class="n">2</span><div class="t">Existing Lakebase table</div><div class="w">Columns already there &mdash; just index</div></label>
    <label class="btn" for="lbss-m3"><span class="n">3</span><div class="t">Sync from Unity Catalog</div><div class="w">Data lives in UC &mdash; sync it in</div></label>
  </div>
  <div class="panels">
    <div class="panel" id="lbss-p1">
      <div class="step"><div class="sn">1</div><div class="sd">Create a new table with <span class="mono">VECTOR(n)</span> and <span class="mono">TSVECTOR</span> columns.</div></div>
      <div class="step"><div class="sn">2</div><div class="sd">Insert your data directly into the table.</div></div>
      <div class="step"><div class="sn">3</div><div class="sd">Build the <span class="mono" style="background:#FFF0ED;color:#C0341D;">lakebase_ann</span> and <span class="mono" style="background:#E7EEF1;color:#1B5162;">lakebase_bm25</span> indexes.</div></div>
    </div>
    <div class="panel" id="lbss-p2">
      <div class="step"><div class="sn">1</div><div class="sd">You already have a Lakebase table with the proper <span class="mono">VECTOR</span> and <span class="mono">TSVECTOR</span> columns.</div></div>
      <div class="step"><div class="sn">2</div><div class="sd">Just create the <span class="mono" style="background:#FFF0ED;color:#C0341D;">lakebase_ann</span> and <span class="mono" style="background:#E7EEF1;color:#1B5162;">lakebase_bm25</span> indexes on the existing columns.</div></div>
    </div>
    <div class="panel" id="lbss-p3">
      <div class="step"><div class="sn">1</div><div class="sd">Your embeddings and source text live in a <b>Unity Catalog</b> table.</div></div>
      <div class="step"><div class="sn">2</div><div class="sd">Use <b>synced tables</b> with custom type mappings: map the embedding column to <span class="mono">VECTOR(n)</span> and generate a <span class="mono">TSVECTOR</span> column from a source text column during the sync.</div></div>
      <div class="step"><div class="sn">3</div><div class="sd">Build the <span class="mono" style="background:#FFF0ED;color:#C0341D;">lakebase_ann</span> and <span class="mono" style="background:#E7EEF1;color:#1B5162;">lakebase_bm25</span> indexes on the synced columns.</div></div>
    </div>
  </div>
</div>



## B. Querying Lakebase Tables

More generally, when querying Lakebase tables with a managed SQL editor on Databricks, there are two options: 
- **Lakehouse SQL Editor**
    - Navigate to the left side menu and, in a new tab, open the SQL Editor. 
        1. **Select the compute resource**: Click on the compute dropdown at the top right and select **More...**
        2. Under **Compute type**, select **Lakebase Postgres**
        3. Select **Autoscaling** and choose the Lakebase project name shown in the output table from running the classroom setup script above
        4. Click **Attach**
        5. Begin querying

- **Lakebase SQL Editor**
    - Navigate to the waffle at the top right and click on **Lakebase Postrgres**. 
        1. Locate your project by searching for the same project name that is shown in the output of the classroom setup. Be sure to leverage the copy button provided. 
        2. Click on **Tables**. You will see the table **doc_chunks**. This table is actually a snapshot of a volume that has been indexed via enabling **Content Search**.  

You can choose either **SQL Editor** in what follows.

### B1. Querying Lakebase Search

With the data loaded and indexed by the classroom setup, the next three blocks provide copyable Lakebase SQL versions of the search workloads against the **doc_chunks** table.

Instead of using an existing chunk as the probe, we will use a natural-language query that matches the Airbnb data loaded into the table. A good example from this dataset is:

`I want a Victorian house in Hayes Valley for a month-long stay.`

That question should surface content about the Victorian listings that mention Hayes Valley and minimum one-month stays.

<div class="scn">
  <style>
    .scn { max-width: 1280px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; color: #0b2026; }
    .scn svg { display: block; width: 100%; height: auto; }
  </style>
  <svg viewBox="0 0 1280 384" role="img" aria-label="User query is embedded outside the database, the vector is passed into Lakebase Postgres for ANN search, and results return to the user.">
    <defs>
      <marker id="scn-ar-g" viewBox="0 0 10 10" refX="0" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M0 0 L10 5 L0 10 z" fill="#618794"/></marker>
      <marker id="scn-ar-l" viewBox="0 0 10 10" refX="0" refY="5" markerUnits="userSpaceOnUse" markerWidth="12" markerHeight="12" orient="auto"><path d="M0 0 L10 5 L0 10 z" fill="#FF5F46"/></marker>
      <marker id="scn-ar-n" viewBox="0 0 10 10" refX="0" refY="5" markerUnits="userSpaceOnUse" markerWidth="11" markerHeight="11" orient="auto"><path d="M0 0 L10 5 L0 10 z" fill="#1B5162"/></marker>
    </defs>
    <!-- ================================================================
         LAYOUT GRID
         card row:      y 100 -> 244  (center y = 172)
         zones:         y 56  -> 268
         cards 1-3:     192w at x 160 / 402 / 644  (uniform 50px gaps)
         gap 1: 352 -> 402, center 377  | arrow line 358 -> 389, tip 400
         gap 2: 594 -> 644, center 619  | arrow line 600 -> 631, tip 642
         outside zone:  x 136 -> 860
         inside zone:   x 960 -> 1264
         all h-arrows:  y 172
         return loop:   y 332
       ================================================================ -->
    <!-- ===== User ===== -->
    <text x="56" y="110" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">User</text>
    <circle cx="56" cy="142" r="13" fill="#1B5162"/>
    <path d="M33 192 C33 172 43 159 56 159 C69 159 79 172 79 192 Z" fill="#1B5162"/>
    <line x1="88" y1="172" x2="114" y2="172" stroke="#618794" stroke-width="2.4" marker-end="url(#scn-ar-g)"/>
    <!-- ===== OUTSIDE zone ===== -->
    <rect x="136" y="56" width="724" height="212" rx="14" fill="#F8F9FC" stroke="#C4CCD6" stroke-width="1.5" stroke-dasharray="6 5"/>
    <text x="160" y="84" font-size="14" font-weight="800" letter-spacing="0.6" fill="#5A6F77">YOUR APP &#183; NOTEBOOK &#183; PIPELINE</text>
    <!-- box 1: x 160 -> 352, center 256 -->
    <rect x="160" y="100" width="192" height="144" rx="10" fill="rgba(66,153,224,0.12)" stroke="#4299E0" stroke-width="1.5"/>
    <circle cx="256" cy="132" r="13" fill="rgba(66,153,224,0.22)"/><text x="256" y="137" text-anchor="middle" font-size="14" font-weight="800" fill="#2272B4">1</text>
    <text x="256" y="174" text-anchor="middle" font-size="16" font-weight="800" fill="#0b2026">User query</text>
    <text x="256" y="202" text-anchor="middle" font-size="14" fill="#618794">plain text</text>
    <!-- arrow 1 -> 2 : centered in gap 352-402 -->
    <line x1="358" y1="172" x2="389" y2="172" stroke="#618794" stroke-width="2.4" marker-end="url(#scn-ar-g)"/>
    <!-- box 2: x 402 -> 594, center 498 -->
    <rect x="402" y="100" width="192" height="144" rx="10" fill="rgba(0,169,114,0.12)" stroke="#00A972" stroke-width="1.5"/>
    <circle cx="498" cy="132" r="13" fill="rgba(0,169,114,0.22)"/><text x="498" y="137" text-anchor="middle" font-size="14" font-weight="800" fill="#00A972">2</text>
    <text x="498" y="172" text-anchor="middle" font-size="16" font-weight="800" fill="#0b2026">Embed the text</text>
    <text x="498" y="196" text-anchor="middle" font-size="14" fill="#618794">text &#8594; 1024-float vector</text>
    <rect x="412" y="208" width="172" height="24" rx="6" fill="rgba(0,169,114,0.18)"/>
    <text x="498" y="225" text-anchor="middle" font-size="14" font-family="ui-monospace,Menlo,Consolas,monospace" fill="#0b2026">ai_query() &#183; API</text>
    <!-- arrow 2 -> 3 : centered in gap 594-644 -->
    <line x1="600" y1="172" x2="631" y2="172" stroke="#618794" stroke-width="2.4" marker-end="url(#scn-ar-g)"/>
    <!-- box 3: x 644 -> 836, center 740 -->
    <rect x="644" y="100" width="192" height="144" rx="10" fill="rgba(255,171,0,0.14)" stroke="#FFAB00" stroke-width="1.5"/>
    <circle cx="740" cy="132" r="13" fill="rgba(255,171,0,0.26)"/><text x="740" y="137" text-anchor="middle" font-size="14" font-weight="800" fill="#0b2026">3</text>
    <text x="740" y="174" text-anchor="middle" font-size="16" font-weight="800" fill="#0b2026">Pass vector to SQL</text>
    <text x="740" y="202" text-anchor="middle" font-size="14" fill="#618794">the literal to compare</text>
    <!-- ===== boundary arrow: gap 860 -> 960, centered ===== -->
    <line x1="878" y1="172" x2="930" y2="172" stroke="#FF5F46" stroke-width="3.2" marker-end="url(#scn-ar-l)"/>
    <!-- ===== INSIDE zone ===== -->
    <rect x="960" y="56" width="304" height="212" rx="14" fill="#FFF6F4" stroke="#FF5F46" stroke-width="2"/>
    <text x="984" y="84" font-size="14" font-weight="800" letter-spacing="0.6" fill="#FF5F46">LAKEBASE (POSTGRES)</text>
    <!-- ANN box: x 984 -> 1240, center 1112 -->
    <rect x="984" y="100" width="256" height="144" rx="10" fill="rgba(255,95,70,0.12)" stroke="#FF5F46" stroke-width="1.5"/>
    <circle cx="1112" cy="132" r="13" fill="#FF5F46"/><text x="1112" y="137" text-anchor="middle" font-size="14" font-weight="800" fill="#FFFFFF">4</text>
    <text x="1112" y="172" text-anchor="middle" font-size="16" font-weight="800" fill="#0b2026">ANN search</text>
    <text x="1112" y="196" text-anchor="middle" font-size="14" fill="#618794">returns nearest rows</text>
    <rect x="1026" y="208" width="172" height="24" rx="6" fill="rgba(255,95,70,0.22)"/>
    <text x="1112" y="225" text-anchor="middle" font-size="14" font-family="ui-monospace,Menlo,Consolas,monospace" fill="#98102A">lakebase_ann</text>
    <!-- ===== return loop ===== -->
    <path d="M1112 268 L1112 332 L840 332" fill="none" stroke="#1B5162" stroke-width="2.4"/>
    <path d="M444 332 L56 332 L56 200" fill="none" stroke="#1B5162" stroke-width="2.4" marker-end="url(#scn-ar-n)"/>
    <text x="642" y="338" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162">5 &#183; Results go back to the user</text>
  </svg>
</div>

Here you will use natural language to create an embedding with `ai_query()` and you will pass the resulting embedding using SQL in the **Lakebase SQL Editor**.

Following the diagram from the cell above, we're first going to pass our query to `ai_query()` to render the embedding and save. 

Now, of course in production, we would wire this into an application, perform proper tracing, etc. But this will give you an idea of the workflow using vector search with Lakebase. Copy and paste the following into the SQL block below to generate the embedding. 

<div class="code-card">
      <div class="code-card-header">
        <span class="code-card-lang">Prompt</span>
        <button class="code-card-copy" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.classList.add('copied');setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.classList.remove('copied')},2000)}})(this)">
          <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
            <rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/>
          </svg>
          <svg class="check-icon" style="display:none" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round">
            <polyline points="20 6 9 17 4 12"/>
          </svg>
          <span class="copy-label">Copy</span>
        </button>
      </div>
      <div class="code-card-body">I want a Victorian house in Hayes Valley for a month-long stay.</div>
    </div>

<style>
  .code-card { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; border: 1px solid #DCE0E2; border-radius: 0.5rem; overflow: hidden; background: #F9F7F4; max-width: 100%; margin: 0.5rem 0; }
  .code-card-header { display: flex; align-items: center; justify-content: space-between; padding: 0.4rem 0.75rem; background: #EEEDE9; border-bottom: 1px solid #DCE0E2; }
  .code-card-lang { font-size: 0.7rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.1em; color: #5A6F77; }
  .code-card-copy { display: inline-flex; align-items: center; gap: 0.3rem; background: none; border: none; color: #5A6F77; cursor: pointer; padding: 0.2rem 0.4rem; border-radius: 0.25rem; font-size: 0.7rem; font-weight: 500; transition: color 0.15s, background 0.15s; }
  .code-card-copy:hover { color: #1B3139; background: rgba(0,0,0,0.05); }
  .code-card-copy.copied { color: #2272B4; }
  .code-card-copy svg { width: 14px; height: 14px; }
  .code-card-body { margin: 0; padding: 1rem; overflow-x: auto; font-family: 'SF Mono', 'Cascadia Code', 'Fira Code', Consolas, monospace; font-size: 0.82rem; line-height: 1.55; color: #1B3139; white-space: pre-wrap; word-break: break-word; }
  .catalog-input-card { display: flex; align-items: center; gap: 14px; flex-wrap: wrap; border: 2px solid #1976d2; background: #e3f2fd; padding: 14px 20px; border-radius: 8px; margin: 16px 0; font-family: -apple-system, 'Segoe UI', Roboto, sans-serif; }
  .catalog-input-card label { font-weight: 700; color: #0d47a1; font-size: 0.95rem; white-space: nowrap; }
  .catalog-input-card input { font-family: 'SF Mono', 'Cascadia Code', Consolas, monospace; font-size: 0.9rem; padding: 8px 14px; border: 2px solid #1976d2; border-radius: 6px; background: #fff; min-width: 280px; color: #1B3139; outline: none; transition: border-color 0.2s, box-shadow 0.2s; }
  .catalog-input-card input:focus { border-color: #0d47a1; box-shadow: 0 0 0 3px rgba(25, 118, 210, 0.15); }
  .catalog-fill { background: #fff3cd; padding: 1px 4px; border-radius: 3px; border-bottom: 2px solid #ffc107; font-weight: 600; }
</style>

You will need to leverage a hosted model within Databricks to intiate `ai_query()`. 

More importantly, the embedding model _must_ match dimensions in our lakebase table. 

1. Navgate back to the **doc_chunks** table and determine the dimensionality of your embedding column **embedding**. What dimension do you see? 

2. Copy and paste the following embedding model into `ai_query()`

The function `ai_query()` takes `(endpoint, request)` as inputs. 


<div class="code-card">
      <div class="code-card-header">
        <span class="code-card-lang">Prompt</span>
        <button class="code-card-copy" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.classList.add('copied');setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.classList.remove('copied')},2000)}})(this)">
          <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
            <rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/>
          </svg>
          <svg class="check-icon" style="display:none" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round">
            <polyline points="20 6 9 17 4 12"/>
          </svg>
          <span class="copy-label">Copy</span>
        </button>
      </div>
      <div class="code-card-body"><code>databricks-qwen3-embedding-0-6b</code></div>
    </div>

Documentation: Please see the following official documentation for `ai_query` [AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_query) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_query)

In [0]:
## TODO

embedding_df = spark.sql(f"""
  SELECT ai_query(
    <FILL_IN>,
    <FILL_IN>
  ) AS embedding
""")

display(embedding_df)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
embedding_df = spark.sql(f"""
  SELECT ai_query(
    'databricks-qwen3-embedding-0-6b',
    'I want a Victorian house in Hayes Valley for a month-long stay.'
  ) AS embedding
""")

display(embedding_df)
```

</div>
</details>

### B2. Save the Embedding
Next, we're going to save this embedding locally for demonstration purposes. 

Of course, you would probably save your embeddings in a table in Unity Catalog or have them computed on-the-fly and stored in ephemeral memory in your app. 

Running the next code will create a file called `embedding.txt` in your Workspace folder. After running the next cell, go to the Workspace folder to the left and inspect the file.

In [0]:
import json

# Collect the embedding from the DataFrame
embedding = embedding_df.collect()[0]["embedding"]

# Save to embedding.txt in this notebook's folder
file_path = "./embedding.txt"

with open(file_path, "w") as f:
    if isinstance(embedding, (list, tuple)):
        f.write(json.dumps(embedding))
    else:
        f.write(str(embedding))

print(f"Embedding saved to: {file_path}")

### B3. Vector Similarity Search

#### NOTE: Run the query in Lakebase

Now let's perform a vector similarity search using the embedding you just saved. 

There are a few operator classes to be familiar with, whose definitions can be found in the docs ([AWS](https://docs.databricks.com/aws/en/oltp/projects/lakebase-vector#operator-classes) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/lakebase-vector)). We will be using the `vector_cosine_ops` class, which computes the cosine similarity between embeddings. Its query operator is denoted by `<=>` and returns a **cosine distance** &mdash; lower values mean higher similarity. We alias this as `distance` so you can see the score alongside each result, just like the `score` column in the BM25 query later.

Fill in the three `FILL_IN` placeholders below:
1. The **embedding column** (`embedding`) on the left side of `<=>`

2. The **1024-dimensional embedding** from `embedding.txt` &mdash; paste it on the right side of `<=>`

3. The **table name** (`doc_chunks`)

4. The output should return a chunk of text along with a distance score of ~ 0.501

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><line x1="1.5" y1="5.5" x2="18.5" y2="5.5" stroke="#5A6F77" stroke-width="1.5"/><path d="M5.5 8.5L8.5 11L5.5 13.5" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="13" x2="14.5" y2="13" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">SQL Query</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
  <div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">SELECT chunk_text, &amp;lt;FILL_IN&amp;gt; &lt;=&gt; '&amp;lt;FILL_IN&amp;gt;' AS distance
FROM &amp;lt;FILL_IN&amp;gt;
ORDER BY distance;</div>
</div>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
SELECT chunk_text, embedding <=> 'paste your 1024-dim embedding from embedding.txt' AS distance
FROM doc_chunks
ORDER BY distance;
```

**Note:** Paste the full 1024-dimensional embedding vector from your `embedding.txt` file in place of the placeholder above. The `distance` column shows cosine distance &mdash; chunks with the **lowest** values are the most semantically similar to your query.

</div>
</details>

### B4. Keyword Search with `lakebase_bm25`

#### NOTE: Run the query in Lakebase

Now let's perform a keyword search using the `lakebase_bm25` index. 

The key operator here is `<@>`, whose definition can be found in the docs ([AWS](https://docs.databricks.com/aws/en/oltp/projects/lakebase-text#operators) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/lakebase-text)). Its signature is `tsvector <@> bm25query_tsvector` and it returns a `double precision` value representing a **negative BM25 score** &mdash; order **ascending** to get the most relevant results first.

Fill in the four `&lt;FILL_IN&gt;` placeholders below:

1. All columns and the BM25 score (`*,`)

2. The **tsvector column** (`chunk_tsvector`) on the left side of `<@>`

3. The **search term** to match against (e.g. `'Victorian'`)

4. The **BM25 index name** (`doc_chunks_ts_idx`)

5. The output should return a chunk of text along with a distance score of ~ -0.556.

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><line x1="1.5" y1="5.5" x2="18.5" y2="5.5" stroke="#5A6F77" stroke-width="1.5"/><path d="M5.5 8.5L8.5 11L5.5 13.5" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="13" x2="14.5" y2="13" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">SQL Query</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
  <div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">SELECT &amp;lt;FILL_IN&amp;gt;,
  &amp;lt;FILL_IN&amp;gt; &lt;@&gt; to_bm25query(to_tsvector('english', '&amp;lt;FILL_IN&amp;gt;'), '&amp;lt;FILL_IN&amp;gt;') AS score
FROM doc_chunks
ORDER BY score;</div>
</div>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
SELECT *,
  chunk_tsvector <@> to_bm25query(to_tsvector('english', 'Victorian'), 'doc_chunks_ts_idx') AS score
FROM doc_chunks
ORDER BY score;
```

</div>
</details>

### B5. Hybrid Search

#### NOTE: Run the query in Lakebase

Finally, we will perform a hybrid search. 

It retrieves the top candidates from each search _independently_, then combines them into a single ranking using **Reciprocal Rank Fusion (RRF)**. Results that rank well in either or both searches score higher. You can read more here ([AWS](https://docs.databricks.com/aws/en/oltp/projects/lakebase-search#combine-results-with-hybrid-search) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/lakebase-search)).

Using what you learned in B3 and B4, fill in the four `&lt;FILL_IN&gt;` placeholders below:
1. The **1024-dimensional embedding** from `embedding.txt` &mdash; paste it on the right side of `<=>`

2. The **tsvector column** (`chunk_tsvector`) on the left side of `<@>`

3. The **search term** (e.g. `'Victorian'`)

4. The **BM25 index name** (`doc_chunks_ts_idx`)

5. You will see the `rrf_score` column, ranked from best search results to least. 

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="20" height="17" viewBox="0 0 20 17" fill="none" style="display:inline-block;vertical-align:middle;"><rect x="1.5" y="1" width="17" height="15" rx="2.5" stroke="#5A6F77" stroke-width="2"/><line x1="1.5" y1="5.5" x2="18.5" y2="5.5" stroke="#5A6F77" stroke-width="1.5"/><path d="M5.5 8.5L8.5 11L5.5 13.5" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/><line x1="10.5" y1="13" x2="14.5" y2="13" stroke="#5A6F77" stroke-width="1.8" stroke-linecap="round"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">SQL Query</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
  <div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">WITH vector_ranked AS (
  SELECT chunk_id, RANK() OVER (ORDER BY dist) AS rank
  FROM (
    SELECT chunk_id, embedding &lt;=&gt; '&amp;lt;FILL_IN&amp;gt;' AS dist
    FROM doc_chunks
    ORDER BY dist
    LIMIT 40
  ) v
),
keyword_ranked AS (
  SELECT chunk_id, RANK() OVER (ORDER BY score) AS rank
  FROM (
    SELECT chunk_id,
      &amp;lt;FILL_IN&amp;gt; &lt;@&gt; to_bm25query(to_tsvector('english', '&amp;lt;FILL_IN&amp;gt;'), '&amp;lt;FILL_IN&amp;gt;') AS score
    FROM doc_chunks
    ORDER BY score
    LIMIT 40
  ) k
)
SELECT d.chunk_id, LEFT(d.chunk_text, 120) AS preview,
  COALESCE(1.0 / (60 + v.rank), 0) + COALESCE(1.0 / (60 + k.rank), 0) AS rrf_score
FROM doc_chunks d
LEFT JOIN vector_ranked v ON d.chunk_id = v.chunk_id
LEFT JOIN keyword_ranked k ON d.chunk_id = k.chunk_id
WHERE v.chunk_id IS NOT NULL OR k.chunk_id IS NOT NULL
ORDER BY rrf_score DESC, d.chunk_id
LIMIT 10;</div>
</div>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
WITH vector_ranked AS (
  SELECT chunk_id, RANK() OVER (ORDER BY dist) AS rank
  FROM (
    SELECT chunk_id, embedding <=> '<paste your 1024-dim embedding from embedding.txt>' AS dist
    FROM doc_chunks
    ORDER BY dist
    LIMIT 40
  ) v
),
keyword_ranked AS (
  SELECT chunk_id, RANK() OVER (ORDER BY score) AS rank
  FROM (
    SELECT chunk_id,
      chunk_tsvector <@> to_bm25query(to_tsvector('english', 'Victorian'), 'doc_chunks_ts_idx') AS score
    FROM doc_chunks
    ORDER BY score
    LIMIT 40
  ) k
)
SELECT d.chunk_id, LEFT(d.chunk_text, 120) AS preview,
  COALESCE(1.0 / (60 + v.rank), 0) + COALESCE(1.0 / (60 + k.rank), 0) AS rrf_score
FROM doc_chunks d
LEFT JOIN vector_ranked v ON d.chunk_id = v.chunk_id
LEFT JOIN keyword_ranked k ON d.chunk_id = k.chunk_id
WHERE v.chunk_id IS NOT NULL OR k.chunk_id IS NOT NULL
ORDER BY rrf_score DESC, d.chunk_id
LIMIT 10;
```

**Note:** Paste the full 1024-dimensional embedding vector from your `embedding.txt` file in place of the placeholder above.

</div>
</details>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>